# TartanIMU Challenge — Starter Notebook

From a 1.0 s window of 6-axis IMU (accel + gyro, 200 Hz, body frame, gravity
retained), predict the sensor's 3-D **body-frame velocity** `(vx, vy, vz)`.
One model, four embodiments: **car / dog / drone / human**.

Leaderboard metric = **macro-averaged 20 m-segment ATE**, in metres, lower is
better. You submit per-window velocities; the organizers integrate them with
ground-truth orientation inside each ~20 m stretch of the true path, align each
segment with Umeyama SE(3) (no scale), and take the RMS position error —
averaged over segments, then trajectories, then the four platforms equally.
**You submit velocity, you are scored on displacement.**

This notebook walks through: install → inspect data → all-zero baseline →
pretrained TartanIMU baseline → submit. Set `DATA` to where you unzipped the
competition data.

In [ ]:
# 1. Install the library (run once)
# !git clone https://github.com/superxslam/TartanIMU && cd TartanIMU && pip install -e .
# !pip install huggingface_hub

import os
import csv
import glob

import numpy as np

# Local unzip: point DATA at the folder holding train/ val/ test/ index/.
DATA = "/path/to/kaggle/tartanimu"   # <-- edit me

# On Kaggle Notebooks the data is already mounted — this finds it without
# hard-coding the competition slug.
_mounted = [d for d in glob.glob("/kaggle/input/*") + glob.glob("/kaggle/input/*/*")
            if os.path.isdir(os.path.join(d, "index"))]
if _mounted:
    DATA = _mounted[0]

TEST_ROOT = os.path.join(DATA, "test")
TEST_WINDOWS = os.path.join(DATA, "index", "test_windows.csv")
VAL_WINDOWS = os.path.join(DATA, "index", "val_windows.csv")
SAMPLE_SUB = os.path.join(DATA, "sample_submission.csv")

print("DATA =", DATA)
print(sorted(os.listdir(DATA)))

## Inspect the data

`train/` and `val/` are organized by platform and carry full ground truth
(`imu, ts, pos, quat, vel_body, platform_id, fs`). `test/` is **anonymized**:
flat `test_0000.npz` … `test_0088.npz`, inputs only (`imu, ts, fs`), no platform
label anywhere — a single model has to work without being told the embodiment.

A window is 200 frames = 1.0 s, and windows are **non-overlapping**: window `k`
of a trajectory is `imu[k*200:(k+1)*200]`.

In [ ]:
sample = sorted(f for f in os.listdir(TEST_ROOT) if f.endswith('.npz'))[0]
npz = np.load(os.path.join(TEST_ROOT, sample))
imu = npz['imu']
print(sample, 'keys', list(npz.keys()))
print('imu shape', imu.shape, '-> ~', imu.shape[0] // 200, 'windows')
print('accel mean', imu[:, :3].mean(0), 'gyro mean', imu[:, 3:].mean(0))

# Gravity is retained — the accelerometer norm sits near 9.81.
print('mean |accel| = %.2f m/s^2' % np.linalg.norm(imu[:, :3], axis=1).mean())

# The test index carries no platform column; the labelled ones do.
print('\ntest index columns:', next(iter(csv.reader(open(TEST_WINDOWS)))))

## Baseline 1 — all-zero submission (verify the pipeline)

Fastest way to confirm your submission and the leaderboard work: correct id
space, correct columns, all zeros. It scores **3.058** on the public split.

Note you cannot win by predicting zero — the segment metric is built so that
standing still loses on every platform.

In [ ]:
with open(SAMPLE_SUB) as f:
    wids = [r['window_id'] for r in csv.DictReader(f)]
with open('submission_zero.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['window_id', 'vx', 'vy', 'vz'])
    for wid in wids:
        w.writerow([wid, 0.0, 0.0, 0.0])
print('wrote submission_zero.csv with', len(wids), 'rows')   # expect 30644

## Baseline 2 — released pretrained TartanIMU model

Downloads the released unified 4-head model from the Hugging Face Hub
([`Tartan-IMU/TartanIMU`](https://huggingface.co/Tartan-IMU/TartanIMU)) and runs
it over a split.

**On the labelled `val` split** the windows CSV has a `platform` column, so the
script routes heads automatically — this is the way to reproduce numbers and to
self-score with `kaggle_metric_ate20.py`.

**On the competition `test` split** there is no platform column, so a multi-head
model has no routing signal: pass `--head` to force one head, or `--routing` to
supply your own per-trajectory choice. Inferring the embodiment inside your
network is allowed and is the point of the benchmark; recovering the platform to
dispatch to four separately-trained experts is against the rules.

In [ ]:
# Labelled val split — heads route from the CSV's platform column.
!python tartanimu_submission.py \
    --test_root "$DATA/val" \
    --windows   "$VAL_WINDOWS" \
    --out       submission_val.csv
# add --device cpu if you have no GPU

In [ ]:
# Anonymized test split — no platform column, so a head must be named.
!python tartanimu_submission.py \
    --test_root "$TEST_ROOT" \
    --windows   "$TEST_WINDOWS" \
    --head      human \
    --out       submission_tartanimu.csv

## Submit

Upload `submission_zero.csv` or your own predictions on the competition's
**Submit Predictions** page. The public leaderboard scores on the Public split;
the final ranking uses the held-out Private split.

`kaggle_metric_ate20.py` in this folder is the exact scoring code the
leaderboard runs — use it to self-score on the labelled `val` split while
iterating.

Reference points on the leaderboard metric (Public / Private): ground-truth
velocities **0.162 / 0.147** (the floor), released unified baseline
**1.587 / 1.048**, all-zeros **3.058 / 3.103**, per-platform mean velocity
**3.154 / 3.870**.

Two things that will save you time:

1. `platform_id` in train/val is **0-based** (`car=0, dog=1, drone=2, human=3`).
   The released model's `motion_type` API is **1-based** — add 1 when passing a
   dataset `platform_id` into that model.
2. If you filter windows by ground-truth speed, **raise the limit for drone**. A
   5 m/s gate silently discards the fast half of the drone data, and drone is the
   hardest platform on this leaderboard.